In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import pandas as pd
import time
import os
os.environ['PATH'] += ':/path/to/example/static/Multiwfn_3.8_dev_bin_Linux'

In [ ]:

MAX_TASKS = 2

In [ ]:

NPROC = 10
MEM = 20  

In [ ]:
directory = "Gaussian/opt+freq"

os.makedirs("Gaussian/opt+freq/imaginary_frequency", exist_ok=True)
imaginary_frequency_path = os.path.join(directory, "imaginary_frequency")
failure_path = os.path.join(directory, "failure")
success_path = os.path.join(directory, "success")

In [ ]:
key_words_for_imaginary_frequency = "# opt=(calcall,maxstep=3,notrust) freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682"

In [ ]:

def check_for_imaginary_frequencies(success_path, imaginary_frequency_path):
    
    freq_pattern = re.compile(r"Frequencies\s+--\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)?\s+(-?\d+\.\d+)?")
    
    
    imaginary_freq_count = 0 
    total_files = 0
    
    
    for filename in os.listdir(success_path):
        if filename.endswith(".out"):
            file_path = os.path.join(success_path, filename)
            total_files += 1  
            with open(file_path, 'r') as file:
                
                content = file.read()

                
                frequencies = freq_pattern.findall(content)

                
                has_imaginary_freq = any(float(freq) < 0 for freqs in frequencies for freq in freqs if freq)

                if has_imaginary_freq:
                    
                    base_filename = os.path.splitext(filename)[0]
                    imaginary_freq_count += 1  
                    for ext in ['.out', '.chk', '.gjf']:
                        file_to_move = base_filename + ext
                        src_file_path = os.path.join(success_path, file_to_move)
                        if os.path.exists(src_file_path):
                            shutil.move(src_file_path, imaginary_frequency_path)
                            print(f"Moved {file_to_move} to imaginary frequency folder.")

    
    imaginary_freq_rate = (imaginary_freq_count / total_files) * 100 if total_files > 0 else 0
    
    print("Finished checking for imaginary frequencies.")
    print(f"Number of imaginary_freq calculations: {imaginary_freq_count} ({imaginary_freq_rate:.2f}%)")
    print(f"Total number of files: {total_files}")

In [ ]:

def convert_out_to_gjf(out_file, gjf_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "gjf", "-O", gjf_file])

In [ ]:

def extract_charge_and_coordinates(gjf_file):
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index - 1
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index]
    coordinates_lines = lines[start_index + 1 : final_index]        
    
    return charge_and_multiplicity_line, coordinates_lines

In [ ]:

def extract_smd_parameters(gjf_file):
    smd_parameters = []
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    smd_start_found = False
    for line in lines:
        if line.strip().startswith('eps='):
            smd_start_found = True
        if smd_start_found:
            smd_parameters.append(line)
            if line.strip().startswith('ElectronegativeHalogenicity='):
                break
    
    return smd_parameters

In [ ]:

def modify_and_append_to_gjf(gjf_file, new_keywords, chk_directory, charge_and_coords, smd_parameters):
    print(f"Using keywords for {gjf_file}: {new_keywords}")  
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, gjf_file.replace('.gjf', '.chk'))
    with open(gjf_file, 'w') as file:
        file.write(f"%nprocshared={NPROC}\n")
        file.write(f"%mem={MEM}GB\n")
        file.write(f'%chk={chk_file}\n')
        file.write(f'# {new_keywords}\n\n')
        file.write(f'{gjf_file} energy ECW\n\n')
        file.write(charge_and_coords[0])  
        file.writelines(charge_and_coords[1])  
        file.writelines(smd_parameters)  
        file.write('\n\n')

In [ ]:

def run_gaussian_and_categorize(gjf_file, success_dir, failure_dir):
    base_name = os.path.basename(gjf_file).replace(".gjf", "")
    out_file = base_name + ".out"
    chk_file = base_name + ".chk"
    
    
    try:
        subprocess.run(["g16", gjf_file, out_file])

        
        with open(out_file, 'r') as file:
            contents = file.read()
            if "Normal termination" in contents:
                
                os.rename(gjf_file, os.path.join(success_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(success_dir, out_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return out_file, True
            else:
                
                os.rename(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(failure_dir, out_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(gjf_file, os.path.join(failure_dir, gjf_file))
        if os.path.exists(out_file):
            os.rename(out_file, os.path.join(failure_dir, out_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False    

In [ ]:

def clean_up_failures(success_path, failure_path):
    
    success_files = set()
    for filename in os.listdir(success_path):
        name, ext = os.path.splitext(filename)
        success_files.add(name)

    
    for filename in os.listdir(failure_path):
        name, ext = os.path.splitext(filename)

        
        if name in success_files:
            for ext in ['.gjf', '.chk', '.out']:
                file_to_delete = os.path.join(failure_path, name + ext)
                if os.path.exists(file_to_delete):
                    os.remove(file_to_delete)
                    print(name)

In [ ]:

def check_and_move_files(directory, success_path, failure_path):
    
    
    moved_to_success = []
    moved_to_failure = []

    
    for file in os.listdir(directory):
        if file.endswith(".out"):  
            file_path = os.path.join(directory, file)
            with open(file_path, 'r') as f:
                content = f.read()
                
                if "Normal termination" in content:
                    
                    target_dir = success_path
                    moved_to_success.append(file)  
                else:
                    
                    target_dir = failure_path
                    moved_to_failure.append(file)  
                
                
                base_filename = os.path.splitext(file)[0]
                for ext in ['.gjf', '.chk', '.out']:
                    src_file = os.path.join(directory, base_filename + ext)
                    if os.path.exists(src_file):  
                        dst_file = os.path.join(target_dir, base_filename + ext)
                        os.rename(src_file, dst_file)
    
    
    print("Moved to success folder:")
    for filename in set(moved_to_success):  
        print(os.path.splitext(filename)[0])

    print("Moved to failure folder:")
    for filename in set(moved_to_failure):  
        print(os.path.splitext(filename)[0])

In [ ]:

def check_and_create_gaussian_database():
    
    home_directory = '/data'
    gaussian_database_path = os.path.join(home_directory, 'Gaussian_database')
    
    optfreq_gaussian_database_path = os.path.join(home_directory, 'Gaussian_database/opt+freq')
    RESPpolymer_database_path = os.path.join(home_directory, 'Gaussian_database/RESPpolymer')
    
    
    if not os.path.exists(gaussian_database_path):
        
        os.makedirs(gaussian_database_path)
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(optfreq_gaussian_database_path) and os.path.exists(RESPpolymer_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_gaussian_database_path):
        os.makedirs(optfreq_gaussian_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    else:
        
        print("Public message removed for release.")
    return gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path

In [ ]:

def read_excel_and_create_lists(filepath):
    
    df = pd.read_excel(filepath)
    
    polymers = df[df['is polymer'] == True]['Name'].astype(str).tolist()
    small_molecules = df[df['is polymer'] == False]['Name'].astype(str).tolist()
    return df, polymers, small_molecules

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:

def find_files_for_molecules(source_directory, destination_directory, molecules_list):
    
    os.makedirs(destination_directory, exist_ok=True)
    
    unique_filenames = get_unique_filenames(source_directory)
    
    not_found_molecules = []
    found_molecules = []
    
    for molecule in molecules_list:
        if molecule in unique_filenames:
            found_molecules.append(molecule)
        else:
            not_found_molecules.append(molecule)
    
    print("Found molecules:", found_molecules)
    
    print("Molecules not found:", not_found_molecules)
    return not_found_molecules, found_molecules

In [ ]:

def copy_newly_calculated_files(optfreq_gaussian_database_path, success_directory, not_found_molecules):
    
    os.makedirs(optfreq_gaussian_database_path, exist_ok=True)
    
    
    success_filenames = get_unique_filenames(success_directory)

    
    for molecule in not_found_molecules:
        if molecule in success_filenames:
            for extension in ['.gjf', '.chk', '.out']:
                source_file = os.path.join(success_directory, molecule + extension)
                destination_file = os.path.join(optfreq_gaussian_database_path, molecule + extension)
                
                shutil.copyfile(source_file, destination_file)
                print("Public message removed for release.")

In [ ]:

def main():
    
    
    start_time = time.time()
    
    
    check_for_imaginary_frequencies(success_path, imaginary_frequency_path)
    print("_____________________________________________________________________________________________________________________")
    
    for out_file in os.listdir(imaginary_frequency_path):
        if out_file.endswith(".out"):
            
            source_file = os.path.join(imaginary_frequency_path, out_file)
            target_file = os.path.join(directory, out_file.replace(".out", ".gjf"))
            
            
            base_filename = os.path.splitext(out_file)[0]

            
            convert_out_to_gjf(source_file, target_file)

            
            charge_and_coords = extract_charge_and_coordinates(target_file)

            
            smd_parameters = extract_smd_parameters(source_file)
            
            
            modify_and_append_to_gjf(target_file, key_words_for_imaginary_frequency, directory, charge_and_coords, smd_parameters)
            
            
            print(f"Imaginary frequency found in {source_file}, using key_words_for_imaginary_frequency.")
            
                          
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        futures = [executor.submit(run_gaussian_and_categorize, os.path.join(directory, f), success_path, failure_path) for f in os.listdir(directory) if f.endswith(".gjf")]
        for future in as_completed(futures):
            pass  

    check_and_move_files(directory, success_path, failure_path)
    print("_____________________________________________________________________________________________________________________")
    
    clean_up_failures(success_path, imaginary_frequency_path)
    
    check_for_imaginary_frequencies(success_path, imaginary_frequency_path)
    print("_____________________________________________________________________________________________________________________")
    
    
    gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path = check_and_create_gaussian_database()
    
    print("_____________________________________________________________________________________________________________________")
    
    df, polymers, molecules = read_excel_and_create_lists('System.xlsx')
    
    
    not_found_molecules, found_molecules = find_files_for_molecules(
        optfreq_gaussian_database_path,
        'Gaussian/opt+freq/success',
        molecules
    )
    
    
    copy_newly_calculated_files(optfreq_gaussian_database_path, 'Gaussian/opt+freq/success', not_found_molecules)
    print("_____________________________________________________________________________________________________________________")
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")


In [ ]:
if __name__ == "__main__":
    main()